<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W6D1_Custom_Attention_SMS_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [ ]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [ ]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [ ]:
# TODO: load and inspect the SMS Spam dataset
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)  # load the parquet dataset into a pandas DataFrame
hf_dataset = Dataset.from_pandas(df)  # convert the DataFrame into a Hugging Face Dataset

TRAIN_START = 0
TRAIN_END = 4000  # TODO: use 4,000 samples for training
VAL_START = 4000  # TODO: begin validation split at 4,000
VAL_END = 5000    # TODO: stop validation split at 5,000

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError('Set TRAIN_END, VAL_START, and VAL_END according to the instructions.')

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [ ]:
# TODO: initialize the tokenizer and padding behavior
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'  # TODO: set to 'gpt2', you can also try 'gpt2-medium' or 'gpt2-large'
if MODEL_NAME is None:
    raise ValueError("Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2').")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # TODO: verify pad token is mapped to eos

In [ ]:
# TODO: complete the tokenization function
TEXT_COLUMN = 'sms'        # TODO: set to 'sms' (the name of the text column in the dataset)
PADDING_STRATEGY = 'max_length'   # TODO: set to 'max_length' it will pad to MAX_SEQ_LEN
TRUNCATION_FLAG = True    # TODO: set to True this will truncate sequences longer than MAX_SEQ_LEN
MAX_SEQ_LEN = 64        # TODO: set to 64 because SMS messages are short

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError('Complete TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, and MAX_SEQ_LEN.')


def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)


# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [ ]:
# TODO: instantiate GPT-2 for sequence classification
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # TODO: set to 2 for spam vs. ham because this is binary classification
if NUM_LABELS is None:
    raise ValueError('Set NUM_LABELS to 2 for binary classification.')

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,  # TODO: confirm pad token id so training does not error out
)

# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [ ]:
# TODO: implement the Attention layer
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5  # TODO: use embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(
            query,  # TODO: multiply query with the transposed key
            key.transpose(-2, -1),
        ) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)  # TODO: use the last dimension
        return torch.matmul(attn, value), attn  # TODO: apply attention weights to values


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # TODO: vocab_size, embed_dim
        self.attn = Attention(embed_dim)               # TODO: pass embed_dim
        self.fc = nn.Linear(embed_dim, num_classes)            # TODO: embed_dim to num_classes

    def forward(self, x):
        embed = self.embedding(x)              # TODO: pass input x
        attn_output, _ = self.attn(embed, embed, embed)  # TODO: self-attention (q=k=v=embed)
        pooled = attn_output.mean(dim=1)       # TODO: mean over sequence dimension
        return self.fc(pooled)                      # TODO: classify pooled representation

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [ ]:
# TODO: preprocess datasets for the custom attention model
ATTN_TEXT_COLUMN = 'sms'  # TODO: set to 'sms'
ATTN_MAX_LEN = 64      # TODO: set to 64
if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError('Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN.')


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

In [ ]:
# TODO: create PyTorch DataLoaders
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn  # TODO: set to train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn    # TODO: set to val_ds_attn
if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError('Assign TRAIN_DATA_FOR_LOADER and VAL_DATA_FOR_LOADER before creating loaders.')


train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)


In [ ]:
# TODO: train the custom attention classifier
vocab_size = tokenizer.vocab_size + len(tokenizer.added_tokens_decoder) # TODO: derive from tokenizer (include added tokens)
embed_dim = 64
num_classes = NUM_LABELS   # TODO: set to 2
learning_rate = 1e-3 # TODO: set to 1e-3
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [ ]:
# TODO: configure evaluation metrics
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')   # TODO: 'accuracy'
precision = evaluate.load('precision')  # TODO: 'precision'
recall = evaluate.load('recall')     # TODO: 'recall'
f1 = evaluate.load('f1')         # TODO: 'f1'


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],  # TODO
        'recall': recall.compute(predictions=preds, references=labels)['recall'],          # TODO
        'f1': f1.compute(predictions=preds, references=labels)['f1'],                      # TODO
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [ ]:
# TODO: evaluate GPT-2 on the validation split
gpt2_preds = []
gpt2_labels = []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])


gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],    # TODO
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],  # TODO
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],          # TODO
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],                      # TODO
}
print('GPT-2 Metrics:', gpt2_metrics)

In [ ]:
# TODO: evaluate the custom attention model
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],    # TODO
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],  # TODO
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],          # TODO
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],                      # TODO
}
print('Attention Model Metrics:', attn_metrics)

# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
In the attention mechanism, Query (Q), Key (K), and Value (V) play distinct but interconnected roles:

*   **Query (Q):** This is the element for which we are currently trying to compute attention. It acts like a 'question' or a 'request' for information. For each element in the input sequence, a query is generated, which is then used to compare against all other elements to determine relevance.

*   **Key (K):** These represent the 'labels' or 'identifiers' of the information available in the sequence. Each element in the sequence generates a key. The query is compared against these keys to determine how relevant each element (key) is to the current query.

*   **Value (V):** These hold the actual content or 'information' associated with each element in the sequence. Once the attention scores are computed by comparing queries with keys, these scores are used as weights to linearly combine the values. The result is a weighted sum of values, where elements more relevant to the query (higher attention scores) contribute more to the output representation.

### 2. Why do we use a scaling factor in the dot-product attention?
The scaling factor, typically $1/\sqrt{d_k}$ (where $d_k$ is the dimension of the key vectors), is crucial for numerical stability in dot-product attention. Here's why:

*   **Prevents Large Dot Products:** As the dimension of the key vectors ($d_k$) increases, the magnitude of the dot products between queries and keys tends to grow. Large dot products can push the `softmax` function into regions where its gradients are extremely small (i.e., saturating the `softmax`), leading to very sharp probability distributions where one value is close to 1 and others are close to 0. This phenomenon is known as the 'softmax collapse'.

*   **Stabilizes Gradients:** When the `softmax` saturates, the gradients become tiny, making it difficult for the model to learn effectively during backpropagation. Multiplying the dot products by $1/\sqrt{d_k}$ normalizes these values, bringing them into a more stable range before applying the `softmax`. This helps maintain a more diverse and less extreme probability distribution, which in turn leads to more stable and effective gradients for training.

*   **Improved Training Performance:** By preventing gradient saturation and promoting more stable `softmax` outputs, the scaling factor significantly improves the training performance and convergence of attention-based models, especially as the model complexity and embedding dimensions increase.

### 3. How does self-attention differ from traditional sequence models like RNNs?
Self-attention fundamentally differs from traditional sequence models like Recurrent Neural Networks (RNNs) in several key aspects:

*   **Processing Style (Parallel vs. Sequential):**
    *   **RNNs:** Process sequences sequentially, one token at a time. The hidden state at each step depends on the previous hidden state and the current input. This inherent sequential nature makes them difficult to parallelize during training.
    *   **Self-Attention (Transformers):** Processes all tokens in a sequence simultaneously. It calculates attention scores between each token and every other token in the sequence independently. This parallelization capability is a major advantage for training on modern hardware (GPUs).

*   **Dependency Capture (Long-Range Dependencies):**
    *   **RNNs:** Struggle to capture long-range dependencies effectively. As the sequence length increases, the information from early tokens has to pass through many recurrent steps, leading to issues like vanishing or exploding gradients and making it hard to retain information over long distances.
    *   **Self-Attention:** Directly models the relationships between any two tokens in a sequence, regardless of their distance. Each token can attend to any other token, allowing it to capture long-range dependencies more effectively and directly. This 'attention distance' is constant for all pairs of tokens.

*   **Efficiency (Computation and Memory):**
    *   **RNNs:** Computationally efficient per step, but the sequential nature means the total computation grows linearly with sequence length. Memory can also be an issue for very long sequences due to storing hidden states.
    *   **Self-Attention:** The computational complexity is typically quadratic with respect to the sequence length (due to calculating attention scores between all pairs of tokens). While this can be a bottleneck for extremely long sequences, the ability to parallelize computations makes it much faster for training and often inference compared to RNNs, especially when $d_k$ is small. Memory usage can also be higher for long sequences due to storing the attention matrix.

In summary, while RNNs process information sequentially and struggle with long-range dependencies, self-attention mechanisms offer parallel processing and more direct modeling of relationships across entire sequences, leading to better performance on complex NLP tasks despite higher theoretical complexity for very long sequences.

### 4. Performance analysis

Based on the evaluation metrics:

**GPT-2 Metrics:**
*   Accuracy: 0.157
*   Precision: 0.111
*   Recall: 0.727
*   F1: 0.193

**Custom Attention Model Metrics:**
*   Accuracy: 0.857
*   Precision: 0.167
*   Recall: 0.007
*   F1: 0.014

**Which model performed better?**
It appears neither model performed exceptionally well on all metrics after this initial limited training. However, looking at the F1 score, which is a harmonic mean of precision and recall and is often a good indicator for imbalanced datasets like spam detection, the **GPT-2 model performed slightly better (0.193 F1)** compared to the Custom Attention Model (0.014 F1).

While the Custom Attention Model has a much higher accuracy (0.857 vs 0.157), its extremely low recall (0.007) and F1 score suggest that it is failing to identify almost all of the positive class (spam messages). This is a common issue where a model might achieve high accuracy by simply predicting the majority class (ham messages) most of the time. The GPT-2 model, despite its lower accuracy, has a significantly higher recall, meaning it is better at catching spam, even if it has a lower precision.

**Trade-offs:**
*   **GPT-2 (Pre-trained Transformer):**
    *   **Pros:** Benefits from extensive pre-training on a massive text corpus, enabling it to capture complex language patterns. Its higher recall and F1 indicate better spam detection capability in this limited setup.
    *   **Cons:** Much larger and computationally more expensive. Requires more resources for training and inference. The warning about the `attention_mask` also indicates potential robustness issues if not handled carefully.

*   **Custom Attention Model (Scratch Implementation):**
    *   **Pros:** Lightweight and simple, making it faster to train and deploy, and requiring fewer computational resources. Good for understanding the core attention mechanism.
    *   **Cons:** Very poor performance in terms of recall and F1, suggesting it struggles to learn the nuances of spam detection effectively with this minimal training. It's likely overfitting to the majority class or simply not learning to identify the minority class.

**Suggested improvement for the custom attention classifier:**

One significant improvement for the custom attention classifier would be to implement a **proper training loop with more epochs and a validation step**, instead of just training for a single batch. Currently, the model has only seen a tiny fraction of the training data. Training over the entire `train_loader` for several epochs, with periodic evaluation on the `val_loader` and saving the best model, would allow it to learn much more effectively. Additionally, incorporating **class weighting** in the `CrossEntropyLoss` or using a **different loss function** (like Focal Loss) could help address the class imbalance, which is likely contributing to its poor recall for the spam class.